# 量子电路 AI 编译器 - Colab GPU 训练一键通
只需依次点击每个代码块的运行按钮（或者点击菜单栏：`代码执行程序` -> `全部运行`）。
**注意：请确保上方菜单栏 `代码执行程序` -> `更改运行时类型` 中已选择 `T4 GPU`。**

In [ ]:
# 1. 环境检测与自动防断联配置
import torch
import IPython

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# 自动向浏览器注入防闲置断联 JS 代码（全自动，您无需任何操作）
display(IPython.display.Javascript('''
function keepAlive() {
  document.querySelector("colab-connect-button")?.click();
  console.log("Auto-reconnect triggered: " + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('\n✅ 防断联机制已自动注入后台！')

In [ ]:
# 2. 安装项目依赖库
!pip install -q qiskit==2.3.1 qiskit-aer==0.17.2 gymnasium==1.2.3 rustworkx
!pip install -q torch_geometric
import torch
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
print('\n✅ 依赖库安装完毕！')

In [ ]:
# 3. 获取最新代码
import os
if not os.path.exists('ZJU-Quantum-Compiler'):
    !git clone https://github.com/qqyyqq812/ZJU-Quantum-Compiler.git
else:
    %cd ZJU-Quantum-Compiler
    !git pull
    %cd ..
%cd ZJU-Quantum-Compiler
print('\n✅ 最新代码准备完毕！')

In [ ]:
# 4. 挂载 Google Drive 并启动全量 GPU 训练
from google.colab import drive
import os

# 挂载云盘（会弹出 Google 授权提示，请点击允许，确保断开连接也不会丢失模型）
drive.mount('/content/drive')
DRIVE_SAVE = '/content/drive/MyDrive/quantum_train/models'
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f'\n✅ 模型存放云盘目录已就绪: {DRIVE_SAVE}')

# 硬链接，将原本的输出目录映射到 Google Drive
!mkdir -p models
!rm -rf models/v9_tokyo20
!ln -s /content/drive/MyDrive/quantum_train/models models/v9_tokyo20

print('\n\n================ 🚀 启动 GPU 训练 ================')
!bash run_train_v9_20Q.sh resume

In [ ]:
# 5. 📉 训练可视化监控 
# (可以在训练进行中，随时点击上方代码块左侧的停止按钮，运行本代码块查看进度，完成后再次运行上方代码块继续训练)
import json, matplotlib.pyplot as plt, numpy as np
import os

DRIVE_SAVE = '/content/drive/MyDrive/quantum_train/models'
history_file = f'{DRIVE_SAVE}/history_v7_ibm_tokyo.json'

if os.path.exists(history_file):
    with open(history_file) as f:
        h = json.load(f)
    s = h['episode_swaps']
    window = 100
    avg = [np.mean(s[max(0,i-window):i+1]) for i in range(len(s))]
    plt.figure(figsize=(12,4))
    plt.plot(avg, label='SWAP (Moving Average)')
    plt.axhline(y=20, color='r', linestyle='--', label='Target SWAP<=20')
    plt.xlabel('Episode'); plt.ylabel('SWAP count')
    plt.legend(); plt.title(f'V9 RL Training Curve (Total: {len(s)} ep)')
    plt.show()
else:
    print('⚠️ 暂未找到训练日志文件，请稍等训练写入历史后再运行。')